# Load and Model data
Loads Base.csv into DuckDB, cleans it, and builds metric views.

In [1]:
# import libraries
import duckdb
import os
import pandas as pd

# fix working directory to the root of the project
repo_root = os.path.dirname(os.path.abspath(os.getcwd()))
os.chdir(repo_root)
print("Current working directory:", os.getcwd())


Current working directory: /Users/winnie/Desktop/BAF/fraud-strategy-simulator


In [2]:
conn = duckdb.connect("data/processed/fraud.duckdb")

## Helper function

In [20]:
# helper function to take a SQL file from the sql directory and execute it against the DuckDB connection
def execute_sql_file(conn, sql_file_name):
    sql_file_path = f"{repo_root}/sql/{sql_file_name}.sql"
    with open(sql_file_path, 'r') as file:
        return conn.execute(file.read())

## Load raw data

In [4]:
execute_sql_file(conn, "01_load_raw")

In [5]:
conn.execute("SELECT COUNT(*) AS n_rows FROM raw_applications").df()

,n_rows
0,1000000


In [6]:
conn.execute("DESCRIBE raw_applications").df()

,column_name,column_type,null,key,default,extra
0,application_id,BIGINT,YES,None,None,None
1,fraud_bool,BIGINT,YES,None,None,None
2,income,DOUBLE,YES,None,None,None
3,name_email_similarity,DOUBLE,YES,None,None,None
4,prev_address_months_count,BIGINT,YES,None,None,None
5,current_address_months_count,BIGINT,YES,None,None,None
6,customer_age,BIGINT,YES,None,None,None
7,days_since_request,DOUBLE,YES,None,None,None
8,intended_balcon_amount,DOUBLE,YES,None,None,None
9,payment_type,VARCHAR,YES,None,None,None


In [7]:
conn.execute("SELECT * FROM raw_applications LIMIT 5").df()

,application_id,fraud_bool,income,name_email_similarity,prev_address_months_count,current_address_months_count,customer_age,days_since_request,intended_balcon_amount,payment_type,...,has_other_cards,proposed_credit_limit,foreign_request,source,session_length_in_minutes,device_os,keep_alive_session,device_distinct_emails_8w,device_fraud_count,month
0,1,0,0.3,0.986506,-1,25,40,0.006735,102.453711,AA,...,0,1500.0,0,INTERNET,16.224843,linux,1,1,0,0
1,2,0,0.8,0.617426,-1,89,20,0.010095,-0.849551,AD,...,0,1500.0,0,INTERNET,3.363854,other,1,1,0,0
2,3,0,0.8,0.996707,9,14,40,0.012316,-1.490386,AB,...,0,200.0,0,INTERNET,22.730559,windows,0,1,0,0
3,4,0,0.6,0.475100,11,14,30,0.006991,-1.863101,AB,...,0,200.0,0,INTERNET,15.215816,linux,1,1,0,0
4,5,0,0.9,0.842307,-1,29,40,5.742626,47.152498,AA,...,0,200.0,0,INTERNET,3.743048,other,0,1,0,0


In [8]:
conn.execute("SELECT COUNT(*) FROM raw_applications GROUP BY fraud_bool").df()

,count_star()
0,988971
1,11029


In [9]:
conn.execute("SELECT MIN(month) AS min_date, MAX(month) AS max_date FROM raw_applications").df()

,min_date,max_date
0,0,7


## Missing values profile

In [10]:
conn.execute("SUMMARIZE raw_applications").df()

,column_name,column_type,min,max,approx_unique,avg,std,q25,q50,q75,count,null_percentage
0,application_id,BIGINT,1,1000000,953281,500000.5,288675.27893234405,258730,497572,750000,1000000,0.0
1,fraud_bool,BIGINT,0,1,2,0.011029,0.10443836491621294,0,0,0,1000000,0.0
2,income,DOUBLE,0.1,0.9,9,0.5626956000000565,0.29034260114469307,0.3,0.6000000000000001,0.8,1000000,0.0
3,name_email_similarity,DOUBLE,1.4345504845275636e-06,0.9999993177937188,1167686,0.4936940949631281,0.2891247999953663,0.22467248377211513,0.49171630083326556,0.7553675530953846,1000000,0.0
4,prev_address_months_count,BIGINT,-1,383,405,16.718568,44.04623003277074,-1,-1,14,1000000,0.0
5,current_address_months_count,BIGINT,-1,428,468,86.587867,88.40659910389658,19,52,130,1000000,0.0
6,customer_age,BIGINT,10,90,10,33.68908,12.025798658444003,20,30,40,1000000,0.0
7,days_since_request,DOUBLE,4.036859788721786e-09,78.45690383509861,888490,1.0257052309958057,5.381834634593072,0.007197372761058294,0.015194082324615664,0.02635147461105043,1000000,0.0
8,intended_balcon_amount,DOUBLE,-15.530554840076814,112.9569276953714,1004919,8.66149853717248,20.23615460624679,-1.1812170245899367,-0.8310940711632423,4.766531532399917,1000000,0.0
9,payment_type,VARCHAR,AA,AE,5,NaN,NaN,NaN,NaN,NaN,1000000,0.0


- Dataset has negative values for null values
- We see `prev_address_months_count`, `current_address_months_count`, `bank_months_count`, `session_length_in_minutes`, `device_distinct_emails_8w`, `intended_balcon_amount` has negative as min value and negative values in these columns denote NULL values
- We can see the null values percentage in the query below for these columns
- Also we see `device_fraud_count` has min, max both 0 and there is only 1 unique value. So, it's a constant column which we can drop

In [11]:
# Check for null values in the raw_applications table as dataset
query = """
SELECT
    COUNT(*) AS total_rows,
    ROUND(100.0 * COUNT(*) FILTER (WHERE prev_address_months_count < 0) / COUNT(*), 2)    AS prev_address_months_count,
    ROUND(100.0 * COUNT(*) FILTER (WHERE current_address_months_count < 0) / COUNT(*), 2) AS current_address_months_count,
    ROUND(100.0 * COUNT(*) FILTER (WHERE bank_months_count < 0) / COUNT(*), 2)            AS bank_months_count,
    ROUND(100.0 * COUNT(*) FILTER (WHERE session_length_in_minutes < 0) / COUNT(*), 2)    AS session_length_in_minutes,
    ROUND(100.0 * COUNT(*) FILTER (WHERE device_distinct_emails_8w < 0) / COUNT(*), 2)    AS device_distinct_emails_8w,
    ROUND(100.0 * COUNT(*) FILTER (WHERE intended_balcon_amount < 0) / COUNT(*), 2)       AS intended_balcon_amount
FROM raw_applications
"""
conn.execute(query).df().T

,0
total_rows,1000000.00
prev_address_months_count,71.29
current_address_months_count,0.43
bank_months_count,25.36
session_length_in_minutes,0.20
device_distinct_emails_8w,0.04
intended_balcon_amount,74.25


Missing values are coded as negative numbers. Three columns are heavily missing: `intended_balcon_amount` (74.25%), `prev_address_months_count` (71.29%) and `bank_months_count` (25.36%). These are kept, with missing-indicator flags added, since missingness may be predictive. `current_address_months_count` (0.43%), `session_length_in_minutes` (0.20%) and `device_distinct_emails_8w` (0.04%) have minimal missingness. `device_fraud_count`: [constant at 0 / not constant].

## Clean View

In [12]:
execute_sql_file(conn, "02_clean_view")

In [13]:
conn.execute("""
SELECT
    COUNT(*)                                AS n_rows,
    COUNT(*) - COUNT(prev_address_months_count) AS prev_address_nulls,
    COUNT(*) - COUNT(bank_months_count)         AS bank_months_nulls,
    COUNT(*) - COUNT(intended_balance_amount)   AS intended_balance_nulls,
    SUM(prev_address_missing)               AS prev_address_flags,
    COUNT(DISTINCT channel)                 AS n_channels
FROM applications_clean
""").df().T

,0
n_rows,1000000.0
prev_address_nulls,712920.0
bank_months_nulls,253635.0
intended_balance_nulls,742523.0
prev_address_flags,712920.0
n_channels,2.0


In [14]:
conn.execute("DESCRIBE applications_clean").df()

,column_name,column_type,null,key,default,extra
0,application_id,BIGINT,YES,None,None,None
1,is_fraud,BIGINT,YES,None,None,None
2,month,BIGINT,YES,None,None,None
3,channel,VARCHAR,YES,None,None,None
4,device_os,VARCHAR,YES,None,None,None
5,age_group,BIGINT,YES,None,None,None
6,prev_address_months_count,BIGINT,YES,None,None,None
7,current_address_months_count,BIGINT,YES,None,None,None
8,bank_months_count,BIGINT,YES,None,None,None
9,session_length_in_minutes,DOUBLE,YES,None,None,None


## Metric views

In [15]:
execute_sql_file(conn, "03_metric_views")

In [16]:
conn.execute("SELECT * FROM metrics_overall").df()

,applications,fraud_count,fraud_rate_pct
0,1000000,11029.0,1.1


In [17]:
conn.execute("SELECT * FROM metrics_by_month").df()

,month,applications,fraud_count,fraud_rate_pct
0,0,132440,1500.0,1.13
1,1,127620,1198.0,0.94
2,2,136979,1198.0,0.87
3,3,150936,1392.0,0.92
4,4,127691,1452.0,1.14
5,5,119323,1411.0,1.18
6,6,108168,1450.0,1.34
7,7,96843,1428.0,1.47


In [18]:
conn.execute("SELECT * FROM metrics_by_segment").df()

,segment_type,segment_value,applications,fraud_count,fraud_rate_pct,share_of_applications_pct
0,age_group,90,76,4.0,5.26,0.01
1,age_group,80,1297,64.0,4.93,0.13
2,age_group,70,6517,263.0,4.04,0.65
3,age_group,60,34770,1149.0,3.30,3.48
4,age_group,50,140353,2805.0,2.00,14.04
5,age_group,40,238712,2876.0,1.20,23.87
6,age_group,30,311433,2589.0,0.83,31.14
7,age_group,20,245855,1205.0,0.49,24.59
8,age_group,10,20987,74.0,0.35,2.10
9,channel,phone,7048,112.0,1.59,0.70


## Sanity checks

In [21]:
execute_sql_file(conn, "04_checks").df()

,check_name,failures
0,monthly counts sum to total,0
1,segment counts sum to total,0
2,segment shares sum to 100%,0
3,fraud rates between 0 and 100,0


### Findings
- Overall fraud rate: 1.10% (11,029 of 1,000,000 applications).
- Drift: fraud rate dips to about 0.9% in months 1–3, then rises steadily to 1.47% in month 7. Monthly fraud counts stay roughly flat (about 1,400–1,450) while application volume falls by about a third, so the rate increase comes from shrinking legitimate volume. The test months (6–7) average 1.40% versus 1.03% in the training months (0–5), so the model will be evaluated under distribution shift.
- Device: Windows has the highest fraud rate (2.47%) and accounts for about 59% of all fraud from 26% of applications.
- Age: fraud rate rises steadily with age, from 0.49% (20s) to 3.30% (60s) and over 4% (70+). This is a fairness risk to check later: a model may decline legitimate older applicants at higher rates.
- Channel: phone applications have a higher fraud rate (1.59% vs. 1.10%) but are only 0.7% of volume.
- Small segments: ages 70+, the phone channel and x11 devices are each under 1% of applications; treat their rates with caution.

In [22]:
conn.close()